# Project 1: Data Analysis Project

## Table of contents <a id='toc0_'></a>
- [Question 1: Inequality in Denmark](#toc1_)
  - [Question 1.1: The Gini Coefficient and the Top 10 Percent Share](#toc1_1_)
  - [Question 1.2: Prediction](#toc1_2_)
  - [Question 1.3: Municipalities](#toc1_3_)
  - [Question 1.4: Extension](#toc1_4_)
- [Question 2: Simulating the Income Distribution](#toc2_)
  - [Question 2.1: The Model](#toc2_1_)
  - [Question 2.2: Simulate the Income Distribution](#toc2_2_)
  - [Question 2.3: Compute the Gini Coefficient](#toc2_3_)
  - [Question 2.4: What Drives Inequality?](#toc2_4_)
  - [Question 2.5: Extension: More Risk](#toc2_5_)

We import the nessesary packages:

In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
from scipy import optimize
from scipy.stats import norm
import matplotlib.pyplot as plt

# APIs
from dstapi import DstApi

# plotting
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# functions
import dataproject

## Question 1: Inequality in Denmark <a id='toc1_'></a>

### Question 1.1: The Gini Coefficient and the Top 10 Percent Share <a id='toc1_1_'></a>

We start by getting an overview of the two datasets from Statistic Denmark (DST).

In [ ]:
# Below are some lines of code to explore the datasets. 
# Theses are commented out in the final version of the notebook, 
# but they are used to get a better understanding of the data.

IFOR41 = DstApi('IFOR41') 
IFOR32 = DstApi('IFOR32')

#IFOR41.tablesummary(language='en')
#IFOR32.tablesummary(language='en')

#IFOR41.variable_levels('ULLIG',language='en')
#IFOR41.variable_levels('KOMMUNEDK',language='en')

#IFOR32.variable_levels('DECILGEN',language='en')
#IFOR32.variable_levels('KOMMUNEDK',language='en')



As given by the hint in the assignment description, the Gini coefficient is indicated by `ULLIG = 70`.

The data also consist of a municipality dimension with the 98 municipalities and a `All Denmark = 000`. We load all data about the Gini coefficient from all munipalities from 1987, since later exercises are exploring differences between municipalities.

In [ ]:
# We download both datasets and store them in one inequality dataset:
df_IFOR41 = dataproject.load_IFOR41(ULLIG = '70', KOMMUNEDK = '*', varname='gini')
df_IFOR32 = dataproject.load_IFOR32(DECILGEN = '*', KOMMUNEDK = '*', varname='avg_income')

# Merge the two datasets into one dataset by year and municipality, where year is the index:
df_ineq = pd.merge(df_IFOR32, df_IFOR41, 
                   on = ['year', 'municipality'], 
                   how = 'inner', # Use inner merge, as both datasets should have the same years and municipalities.
                   validate = '1:1')

df_ineq = df_ineq.set_index(['year']).sort_index().sort_values(by=['municipality']) 

# We define a Denmark subset:
denmark = df_ineq[df_ineq['municipality'] == 'All Denmark']

We now plot the Gini coefficient for Denmark over time (1987-2024), as well as the share of total income going to the richest 10 percent. 

In [ ]:
# Makes a Gini and top 10 pct. share plots for Denmark (municipalities = All Denmark) between 1987 and 2024.

fig, ax = plt.subplots(figsize=(12,6)) # Gini
fig, bx = plt.subplots(figsize=(12,6)) # Top 10 pct. share

# Gini coefficient plot
denmark.plot(y = 'gini', 
             ax = ax, 
             color = colors[0], 
             legend = False)

ax.set_xticks(denmark.index.unique()) # Year is set as the index.
ax.set_xticklabels(denmark.index.unique(), rotation=45)

ax.set_title('Gini coefficient in Denmark (1987-2024)', fontsize=16)
ax.set_xlabel('Year', fontsize=14)
ax.set_ylabel('Gini coefficient', fontsize=14)

# Top 10 pct. share plot
denmark.plot(y = 'avg_income_top10_share', 
             ax = bx, 
             color = colors[0], 
                                                       legend = False)

bx.set_xticks(denmark.index.unique()) # Year is set as the index.
bx.set_xticklabels(denmark.index.unique(), rotation=45)

bx.set_title('Top 10% share of total income in Denmark (1987-2024)', fontsize=16)
bx.set_xlabel('Year', fontsize=14)
bx.set_ylabel('Share of total income', fontsize=14)

fig.tight_layout()

The Gini coefficient for Denmark rises over the whole period, indicating a gradual increase in income inequality from 1987 to 2024. At the same time, the share of total income going to the richest 10 percent follows the same pattern, rising alongside the Gini and showing temporary declines during economic downturns such as the dot-com bubble, the financial crisis and the covid-19 pandemic. This suggests that higher inequality is strongly linked to the top income share, and to check this relationship we compute the correlation between the two series over time.

In [ ]:
# We compute the correlation between Gini and top 10% share for Denmark between 1987 and 2024:
corr_gini_top10 = denmark['gini'].corr(denmark['avg_income_top10_share'])
print(f'The correlation between Gini and top 10% share in Denmark (1987-2024) is: {corr_gini_top10:.4f}')

Thus, the Gini coefficient and top 10% share for Denmark almost perfectly correlate between 1987 and 2024. As we saw in the two diagrams, both measures have had similar increases over the time period, with only a slight downturn post the financial crisis.

### Question 1.2: Prediction <a id='toc1_2_'></a>

#### Question 1.2.1 and 1.2.2

To predict inequality in Denmark in 2030 and 2040, we compute the linear and quadratic trend to the Gini coefficient. We do this by first defining the polynomial trend of degree `p`, solving the minimization problem to find the value of beta that best fits the observed data and compare it with `np.plyfit()`.

In [ ]:
#define the polynomial function
def S(beta, t, y, p):
    # beta=[beta_0, beta_1,..., beta_p]
    pred=sum(beta[k]*t**k for k in range(p+1)) #calculating the inner sum
    return np.sum((y-pred)**2)


#compute time, so that it is years since 1987 and define the gini variable as a vector:
time = (denmark.index.values - denmark.index.min())
gini = denmark['gini'].values

# 1. Now, we find the values of beta, beta_hat1 and beta_hat2 that provide the best fit for both the linear and quadratic:
degrees= [1,2]
results={}

print('when using optimize.minimize, the results are:')
for p in degrees:
    initial_guess = np.zeros(p+1) #make the vector
    initial_guess[0] = gini.mean() #and guess that beta is the mean 
    res = optimize.minimize(S, initial_guess, args=(time, gini, p), method='Nelder-Mead')
    results[p] = res

    
    print(f'for p = {p}')
    print(f'  success: {res.success}')
    print(f'  coefficients (betahat_0, ..., betahat_{p}): {res.x}')
    print(f'  S(betahat) at optimum: {res.fun}')
    print()

# 2. We now check the results against np.polyfit:
poly_1=np.polyfit(time, gini, 1)
poly_2=np.polyfit(time, gini, 2)

print('When using np.polyfit, the results are:')
print(f'for p=1, betahat={poly_1}')
print(f'for p=2, betahat={poly_2}')


Both methods agree almost exactly, confirming the manual optimization is correct. The linear fit shows Gini rising about 0.26 points per year since 1987, and the quadratic's small curvature only modestly improves the fit, since $S(\hat{\beta})$ slightly decreases. This suggests a roughly linear trend.

#### Question 1.2.3 

We now use the estimates to predict what the gini coefficient is in 2030 and 2040:

In [ ]:
# Predict the Gini coefficient using the linear and quadratic estimates, and plot the result:
def predict(beta, t, p):
    return sum(beta[k]*t**k for k in range(p+1))

#define the two predictions we want
t_2030 = 2030 - 1987
t_2040 = 2040 - 1987

for p in degrees: 
    beta=results[p].x
    pred_2030=predict(beta, t_2030, p)
    pred_2040=predict(beta, t_2040, p)

    print(f'p={p}')
    print(f'  predicted Gini coefficient in 2030: {pred_2030:.2f}')
    print(f'  predicted Gini coefficient in 2040: {pred_2040:.2f}')
    print()


Now we plot the prediction in a plot as the onces question 1.1.

In [ ]:
# Plot the predicted values with the historic gini data.
fig, cx = plt.subplots(figsize=(12,6))
denmark.plot(y = 'gini', 
             ax = cx, 
             color = colors[0],
             label = 'Observed Gini coefficient')

#Add the predictions:
for p in degrees:
    beta = results[p].x
    pred_2030 = predict(beta, t_2030, p)
    pred_2040 = predict(beta, t_2040, p)
    
    cx.scatter([2030, 2040], [pred_2030, pred_2040],
               color=colors[p], marker='o', s=30, 
               label=f'Predicted Gini coefficient, p={p}')

extra_years = list(range(2024, 2041))
cx.axvline(x=2024, color='black', linestyle='--', linewidth=1)
cx.set_xticks(denmark.index.unique().tolist() + extra_years)
cx.set_xticklabels(cx.get_xticks(), rotation=90)

cx.set_title('Observed Gini coefficient in Denmark (1987-2024), with predictions for 2030 and 2040', fontsize=16)
cx.set_xlabel('Year', fontsize=14)
cx.set_ylabel('Gini coefficient', fontsize=14)
cx.legend()
fig.tight_layout()

So we find that the quadratic trend predicts a slightly higher Gini coefficient in Denmark in 2030 and 2040 than the linear trend, but both predict that the Gini coefficient will dip in 2030, before rising further by 2040. For 2030, the linear trend predict a Gini coefficient of about 29.0 against 29.2 for the quadratic; for 2040, the linear model predicts about 31.3 against about 32.0 for the quadratic. 
We would treat both predictions as rough guess, rather than a reliable forecast. As noted in in 1.1, the Gini coefficient depends on the broader economic situation and on political initatives (e.g. changes to taxation or redistribution). Therefore trend extrpolation may be too simple of a way to predict it. Furthermore, over the observed data, the rise in the Gini coefficient have accelerated after 2004, suggesting that perhaps a better trend prediction would exclude observations prior to this, to give a better prediction.




### Question 1.3: Municipalities <a id='toc1_3_'></a>

1.3.1 We now turn to the gini coefficients for municipalities.

In [ ]:
# Calculating the correlation between the gini and the top 10% share of total income across all observations for each year:
corr_by_year = (
    df_ineq.groupby('year') 
    .apply(lambda x: x['gini'].corr(x['avg_income_top10_share']))
    .reset_index(name='correlation')
)

# And now plot the correlation across years:
plt.figure(figsize=(12,6))
plt.plot(corr_by_year['year'], corr_by_year['correlation'], label='Municipal cross-sectional correlation')
plt.axhline(corr_gini_top10, ls='--', color='black', label='Denmark-wide correlation') 

plt.title('Yearly correlation between Gini and top 10 percent share')
plt.xlabel('Year')
plt.xticks(corr_by_year['year'], rotation=45)
plt.ylabel('Correlation')
plt.ticklabel_format(style='plain', axis='y')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


We find that the cross-sectional correlation across municipalities in a given year is generally lower than the time-series correlation for Denmark as a whole. This is because the national correlation is partly driven by the shared upward trend in both measures over time, whereas the cross-sectional correlation removes this trend and captures only the year-by-year relationship between them. In a single municipality, a small number of high earners can shift the top 10 percent share substantially without moving the Gini coefficient by nearly as much, since Gini reflects the whole distribution while the top 10 percent share only reflects one tail. Therefore, the relationship can be weaker on a year-to-year basis. Municipalities also have far fewer observations than the country as a whole, making both measures noisier.

#### Question 1.3.2 

We now illustrate the 10 most and 10 least unequal municipalities:

In [ ]:
# Finding the top and bottom 10 gini coefficients in 2024:

# Saving only observations from latest year (2024)
df_2024 = df_ineq.loc[2024]
df_2024.sort_values('gini')

df_2024.nlargest(10, 'gini')[['municipality', 'gini']]
df_2024.nsmallest(10, 'gini')[['municipality', 'gini']]

lowest_10 = df_2024.nsmallest(10, 'gini')['municipality'].tolist()
highest_10 = df_2024.nlargest(10, 'gini')['municipality'].tolist()

# The combined list of the 20 municipalities
selected_municipalities = lowest_10 + highest_10

#  Making a dataset of the selected municipalities
df_selected = df_ineq[df_ineq['municipality'].isin(selected_municipalities)]
df_selected['municipality'].unique()

# Plotting the selected 20 municipalities' gini coefs. across the entire period,
# with the top 10 and bottom 10 municipalities in separate side-by-side plots sharing a y-axis.
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

for municipality in highest_10:
    data = df_selected[df_selected['municipality'] == municipality]
    ax_left.plot(data.index, data['gini'], linestyle='-', label=municipality)

for municipality in lowest_10:
    data = df_selected[df_selected['municipality'] == municipality]
    ax_right.plot(data.index, data['gini'], linestyle='-', label=municipality)

ax_left.set_xlabel('Year')
ax_left.set_xticks(df_selected.index.unique())
ax_left.set_xticklabels(df_selected.index.unique(), rotation=45)
ax_left.set_ylabel('Gini coefficient')
ax_left.set_title('10 highest Gini coefficient municipalities in 2024')
ax_left.legend(loc='upper left', fontsize=10, ncol=2)

ax_right.set_xlabel('Year')
ax_right.set_xticks(df_selected.index.unique())
ax_right.set_xticklabels(df_selected.index.unique(), rotation=45)
ax_right.set_title('10 lowest Gini coefficient municipalities in 2024')
ax_right.legend(loc='upper left', fontsize=10, ncol=2)

fig.suptitle('Development in the Gini coefficient for top- and bottom 10 municipalities, 1987-2024', fontsize=14)

fig.tight_layout()
plt.show()

The figure reveals that both the top and bottom 10 municipalities experience an increase in the Gini coefficient throughout the time period. In the late 1980s the municipalities of Copenhagen, Aarhus, Helsingør, and Vejen were on level with the group of the bottom 10 but have since been on a steeper path, compared to the group of the 10 lowest Gini coefficients. An outlier of the data is Vejen whose Gini skyrocketed in 2023 which, according to a quick Google-search, was probably because of one individual selling a large amount of stocks in a leading Danish company.

#### Question 1.3.3

We turn to the municipalities with the largest and smallest changes in the Gini coefficients throughout the time period.

In [ ]:
# Generating a data frame with the Gini coefficients change from the first and last year:  
df_noindex = df_ineq.reset_index()

gini_change = (
    df_noindex[df_noindex['year'].isin([1987, 2024])]
    .pivot(index='municipality', columns='year', values='gini')
)

gini_change['gini_difference'] = (
    gini_change[2024] - gini_change[1987]
)

gini_change = gini_change.sort_values('gini_difference')


# Get the 10 municipalities with the largest and smallest changes
lowest_10 = gini_change.nsmallest(10, 'gini_difference').index.tolist()
highest_10 = gini_change.nlargest(10, 'gini_difference').index.tolist()


selected_municipalities = lowest_10 + highest_10

#  Making a dataset of the selected municipalities
df_selected_2 = df_ineq[
    df_ineq['municipality'].isin(selected_municipalities)
]


# Plotting the selected 20 municipalities' gini coefs, with the largest and smallest
# changes in separate side-by-side plots sharing a y-axis.
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

for municipality in highest_10:
    data = df_selected_2[df_selected_2['municipality'] == municipality]
    ax_left.plot(data.index, data['gini'], linestyle='-', label=municipality)

for municipality in lowest_10:
    data = df_selected_2[df_selected_2['municipality'] == municipality]
    ax_right.plot(data.index, data['gini'], linestyle='-', label=municipality)

ax_left.set_xlabel('Year')
ax_left.set_xticks(df_selected_2.index.unique())
ax_left.set_xticklabels(df_selected_2.index.unique(), rotation=45)
ax_left.set_ylabel('Gini coefficient')
ax_left.set_title('10 municipalities with largest Gini increase')
ax_left.legend(loc='upper left', fontsize=10, ncol=2)

ax_right.set_xlabel('Year')
ax_right.set_xticks(df_selected_2.index.unique())
ax_right.set_xticklabels(df_selected_2.index.unique(), rotation=45)
ax_right.set_title('10 municipalities with smallest Gini increase')
ax_right.legend(loc='upper left', fontsize=10, ncol=2)

fig.suptitle('Development in the Gini coefficient for 10 municipalities with largest and smallest changes over time, 1987-2024', fontsize=14)

fig.tight_layout()
plt.show()

The plot resembles the previous plot, revealing that the municipalities with the smallest changes in Gini form 1987 to 2024 generally are the municipalities with the lowest Gini coefficients in 2024. Likewise the municipalities with the largest increases in Gini are those with the largest Gini coefficients. Furthermore, many of the muncipalities with the largest changes are urban, whereas the smaller changes are in municipalities that are more rural and smaller, like the islands Læsø, Ærø and Fanø. 

### Question 1.4: Extension <a id='toc1_4_'></a>

We now make an extension, and investigate the correlation between the municipalities' Gini coefficient and the municipalities' equalization and grants, fertility and number of criminally convicted people per capita. Our initial hypothesis is that larger income inequality will lead to lower fertility, which makes the correlation negative. Conversely, we would expect that greater income inequality would lead to a higher crime rate, thus a positive correlation between the Gini coefficient and the number of convicted people per capita. Finally, our initial hypothesis on the municipalities' equalization and grants is ambiguous. Richer municipalities could occur either because the general income level is high or because very few people have a very large income.

Thus, we start by exploring and downloading the relevant datasets from DST.

In [ ]:
# Below are some lines of code to explore the datasets. 
# Theses are commented out in the final version of the notebook, 
# but they are used to get a better understanding of the data.

NGLK = DstApi('NGLK') # Municipalities equalization and grants dataset
FOD407 = DstApi('FOD407') # Fertility dataset
STRAFNA7 = DstApi('STRAFNA7') # Crime dataset
BEFOLK3 = DstApi('BEFOLK3') # Population dataset



#NGLK.tablesummary(language='en')
#NGLK.variable_levels('OMRÅDE',language='en') # Needs inner join with IFOR41 since we have few municipalities in IFOR41 and NGLK has fewer years.
#NGLK.variable_levels('BNØGLE',language='en') # Use UDL for Equalization and grants amount per capita
#NGLK.variable_levels('BRUTNETUDG',language='en') # Use NET for net expenditures
#NGLK.variable_levels('PRISENHED',language='en') # Use 08PRIS for Real prices


#FOD407.tablesummary(language='en')
#FOD407.variable_levels('ALDER',language='en') # Use TOT1 for Equalization and grants amount per capita


#STRAFNA7.tablesummary(language='en')
#STRAFNA7.variable_levels('OVERTRÆD',language='en') # Use 1 all crime code 

#BEFOLK3.tablesummary(language='en')
#BEFOLK3.variable_levels('KØN',language='en') # Use TOT for total population
#BEFOLK3.variable_levels('ALDER',language='en') # USE IALT for all ages



In [ ]:
# We download the datasets.
df_equal_muni = dataproject.load_NGLK(OMRÅDE = '*', BNØGLE = 'UDL', BRUTNETUDG = 'NET', PRISENHED = '08PRIS', varname = 'equalization_per_capita')
df_fert_muni = dataproject.load_FOD407(OMRÅDE = '*', ALDER = 'TOT1', varname = 'fertility')
df_crime_muni = dataproject.load_STRAFNA7(OMRÅDE = '*', OVERTRÆD = '1357', varname = 'crime_persons')
df_pop_muni = dataproject.load_BEFOLK3(OMRÅDE = '*', KØN = 'TOT', ALDER = 'IALT', varname = 'population')

# Keep only the Gini column from df_ineq, then merge by year and municipality.
df_corr = df_ineq.reset_index()[['year', 'municipality', 'gini']]

df_corr = (
    df_corr
    .merge(df_equal_muni, on=['year', 'municipality'], how='inner', validate='1:1')
    .merge(df_fert_muni, on=['year', 'municipality'], how='inner', validate='1:1')
    .merge(df_crime_muni, on=['year', 'municipality'], how='inner', validate='1:1')
    .merge(df_pop_muni, on=['year', 'municipality'], how='inner', validate='1:1')
)

# Computes crime rate per capita and deletes crime_persons and population columns
df_corr['crime_rate'] = df_corr['crime_persons'] / df_corr['population']
del df_corr['crime_persons']
del df_corr['population']

df_corr = df_corr.set_index(['year']).sort_index().sort_values(by=['municipality'])

df_corr.head()

Now we plot the correlation between the Gini coeficient and the municipalities equalization and grants, fertility and numbers of guilty persons per capita.

In [ ]:

# Correlations by year between Gini and all three extensions, muncipality equalization, fertility and number of criminally convicted people:
df_ineq_equal_corr = (
    df_corr.groupby('year')
    .apply(lambda x: x['gini'].corr(x['equalization_per_capita']))
    .reset_index(name = 'corr_equalization')
)

df_ineq_fert_corr = (
    df_corr.groupby('year')
    .apply(lambda x: x['gini'].corr(x['fertility']))
    .reset_index(name = 'corr_fertility')
)

df_ineq_crime_corr = (
    df_corr.groupby('year')
    .apply(lambda x: x['gini'].corr(x['crime_rate']))
    .reset_index(name = 'corr_crime_rate')
)

df_corr_plot = df_ineq_equal_corr.merge(
    df_ineq_fert_corr,
    on = 'year',
    how = 'inner'
)

df_corr_plot = df_corr_plot.merge(
    df_ineq_crime_corr,
    on = 'year',
    how = 'inner'
)

# Plot the three correlations in one plot
fig, ax = plt.subplots(figsize=(12, 6))

df_corr_plot.plot(
    x = 'year',
    y = 'corr_equalization',
    ax = ax,
    color = colors[0],
    legend = True,
    label = 'Gini vs municipality equalization per capita'
)

df_corr_plot.plot(
    x = 'year',
    y = 'corr_fertility',
    ax = ax,
    color = colors[1],
    legend = True,
    label = 'Gini vs fertility per 1.000 women'
)

df_corr_plot.plot(
    x = 'year',
    y = 'corr_crime_rate',
    ax = ax,
    color = colors[2],
    legend = True,
    label = 'Gini vs number of guilty persons per capita'
)

ax.set_xticks(df_corr_plot['year'].unique())
ax.set_xticklabels(df_corr_plot['year'].unique(), rotation=45)

ax.set_title('Correlation between Gini and municipality variables (2008-2024)', fontsize=16)
ax.set_xlabel('Year', fontsize=14)
ax.set_ylabel('Correlation', fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()

The blue line (Gini vs equalization per capita) is strongly positive and rises from about 0.49 in 2008 to roughly 0.7, meaning higher inequality goes together with a higher contribution to equalization grants. This resolves the ambiguous hypothesis toward the second explanation: municipalities appear to be unequal mainly because a few residents earn very much. The orange line (fertility) stays negative across the whole period, between about -0.11 and -0.39, which confirms the hypothesis that greater inequality is associated with lower fertility. The green line (guilty persons) is also negative throughout, roughly -0.07 to -0.26, which contradicts the initial hypothesis of a positive crime correlation, so more inequality does not track with more convictions per capita here.

## Question 2: Simulating the Income Distribution <a id='toc2_'></a>

### Question 2.1: The Model <a id='toc2_1_'></a>

In [ ]:
out = dataproject.simulate(seed = 2025, N = 50_000)

# Tests that the simulation produces the correct dimensions
print(f'Shape of ages: {out['ages'].shape} (expected: (48,))')
print(f'Shape of income: {out['income'].shape} (expected: (48, 50000))')
print(f'Last age: {out['ages'][-1]} (expected: 65)')


### Question 2.2: Simulate the Income Distribution <a id='toc2_2_'></a>

#### Question 2.2.1

We check the education shares.

In [ ]:
observed_shares = np.bincount(out['educ']) / len(out['educ'])
expected_shares = [0.40, 0.35, 0.25]
print(f'Observed education shares: {np.round(observed_shares, 2)}')
print(f'Expected education shares: {np.round(expected_shares, 2)}')

We check the unemployment rate.

In [ ]:

# Look at the last period (age 65)
last_t = len(out['ages']) - 1

# Find who is not in education in this period
edu_years_per_person = np.array([1, 3, 5])[out['educ']]
not_in_education = edu_years_per_person <= last_t

unemployment_rate = 1 - out['employed'][last_t][not_in_education].mean()

theoretical = 0.05 / (0.05 + 0.6)  # job_sep_prob_i / (job_sep_prob_i + job_fin_prob)

print(f'Observed unemployment rate: {unemployment_rate:.2%}')
print(f'Theoretical steady-state unemployment rate: {theoretical:.2%}')

We see that the observed unemployment rate is very close to the theoretical steady-state unemployment rate, however still below it. We have tried to increase the numbers of individuals, which leads to an observed umemployment rate that coincides with the theoretical rate. Thus, the mismatch is essentially just Monte Carlo noise from one seed and one finite population, not a modeling error which suggests that the simulation is working as expected.

In [ ]:

# Plots the unemployment rate over time/age
unemp_by_age = 1 - out['employed'].mean(axis=1)  # simple version (ignoring education)
plt.plot(out['ages'], unemp_by_age)
plt.axhline(theoretical, color='red', linestyle='--', label='Theoretical steady-state')

plt.xlabel('Age')
plt.xticks(out['ages'][::5], rotation=45)
plt.ylabel('Unemployment Rate')
plt.legend()
plt.show()


Thus, the observed education shares align closely with the specified probabilities, confirming that the model correctly assigns individuals to education levels. Additionally, the unemployment rate converges to the theoretical steady-state value, validating that the employment dynamics in our simulation are functioning as intended.

#### Question 2.2.2

We plot the mean and selected percentiles of income over the life cycle.

In [ ]:
ages = out['ages']
income = out['income']

mean_income = income.mean(axis=1)
p10 = np.percentile(income, 10, axis=1)
p50 = np.percentile(income, 50, axis=1)
p90 = np.percentile(income, 90, axis=1)

plt.figure(figsize=(8, 5))

plt.fill_between(ages, p10, p90, alpha=0.2, color='steelblue', label='10th-90th percentile')
plt.plot(ages, mean_income, color='steelblue', linewidth=2, label='Mean')
plt.plot(ages, p50, color='steelblue', linestyle='--', label='Median')

plt.xlabel('Age')
plt.xticks(ages[::5], rotation=45)
plt.ylabel('Income')
plt.title('Income over the lifecycle')
plt.legend()
plt.show()

Mean and median income both rise steadily with age, with mean pulling further above median over time, indicating growing right-skew with a few high earners. The 10th–90th percentile band widens continuously from the mid-20s onward, showing that income dispersion across individuals increases substantially over the lifecycle rather than staying constant.

#### Question 2.2.3

We plot a histogram of the income distribution at ages 25, 35, 45, and 60

In [ ]:

target_ages = [25, 35, 45, 60]
target_indices = [a - 18 for a in target_ages] # But the row index starts at 18, so age 25 is at row 7 and so on.

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
xmax = out['income'][target_indices, :].max()

for ax, age, t in zip(axes, target_ages, target_indices):
    ax.hist(out['income'][t, :], bins=100, range=(0, xmax), color='steelblue')
    ax.set_title(f'Age {age}')
    ax.set_xscale('log')
    ax.set_xlabel('Income')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()


At age 25 the distribution is fairly tight and roughly symmetric on the log scale, clustered around 1. As age increases, the distribution shifts right and develops a longer right tail, with more mass spreading out toward higher incomes while the bulk stays concentrated near the lower end. This growing right-skew visually confirms the widening percentile band seen in the lifecycle plot with income dispersion increases with age, driven by a shrinking share of high earners pulling away from the rest.


### Question 2.3: Compute the Gini Coefficient <a id='toc2_3_'></a>

We compute the gini and test if the function is right

In [ ]:

# Test 1: With a uniform distribution on [0, 1] -> 
# Gini should be 1/3
rng = np.random.default_rng(42)
y_uniform = rng.uniform(0, 1, size=1_000_000)
print(f'The uniform distribution test yields a Gini coefficient of {dataproject.gini(y_uniform):.2f}, which is close to the expected value of 0.33.')

# Test 2: With a lognormal distribution 
# -> Gini = 2 * Phi(s/sqrt(2)) - 1
s = 0.5  # std of log income
y_lognormal = rng.lognormal(mean=0, sigma=s, size=1_000_000)
gini_simulated = dataproject.gini(y_lognormal)
gini_theoretical = 2 * norm.cdf(s / np.sqrt(2)) - 1

print(f'Lognormal (simulated): {gini_simulated:.2%}')
print(f'Lognormal (theoretical): {gini_theoretical:.2%}')
print(f'Thus, it is {np.isclose(gini_simulated, gini_theoretical, rtol=0.01)} that the simulated Gini is close to the theoretical Gini.')

#### Question 2.3.1

We compute the Gini coefficient for the full simulated sample and plot the Lorenz curve.

In [ ]:
gini_full_sample = dataproject.plot_full_sample_gini(out['income'], out['ages'])

print(f'The Gini coefficient for the full sample in the last period (age 65) is: {gini_full_sample:.3f}')

#### Question 2.3.2

We compute the by age Gini coefficient.

In [ ]:
# Gini for each age separately:
df_gini_by_age = dataproject.gini_by_age(out['income'], out['ages'])

# Plotting Gini by age
plt.figure(figsize=(8, 5))
plt.plot(df_gini_by_age['age'], df_gini_by_age['gini'], color='steelblue', linewidth=2)
plt.xlabel('Age')
plt.ylabel('Gini coefficient')
plt.title('Gini coefficient by age')
plt.xticks(df_gini_by_age['age'][::5], rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The Gini coefficient starts at 0 at age 18 since everyone are still students and on the same grant. Then jumps sharply as some individuals finish school and enter jobs while others remain in education. Thus, mixing two very different income sources creates strong inequality. It then dips slightly around age 24–25 once most people have finished school and entered the labor force, narrowing the gap between education groups. From there it climbs steadily through the rest of the lifecycle, as compounding human-capital growth and employment/unemployment spells pull incomes further apart with age.

### Question 2.4: What Drives Inequality? <a id='toc2_4_'></a>

We look at the different scenarios to see what drives inequality the most.

In [ ]:
# We define one scenario per mechanism we want to switch off, plus the baseline.
# Each entry only overrides the parameters needed to shut off that specific channel;
# everything else keeps the default values from dataproject.simulate.

scenarios = {
    'Baseline': dict(),

    # Force everyone into education type 0 -> edu_years, ini_hum_cap and gro_hum_cap
    # become identical across the population, so education no longer creates any
    # dispersion in income.
    'No educational differences': dict(edu_prob=[1, 0, 0]),

    # A zero std. means the lognormal shock psi is always exactly 1, i.e. human
    # capital growth/depreciation becomes fully deterministic.
    'No human capital shocks': dict(std_of_shock=0),

    # Human capital no longer shrinks while unemployed.
    'No depreciation while unemployed': dict(depreciation=0),

    # job_fin_prob=1 means everyone is hired the instant they leave education, and
    # job_sep_prob=0 means nobody ever loses their job afterwards. Together this
    # removes unemployment entirely. it is not enough to only set job_sep_prob=0, 
    # since people would still spend time unemployed while searching for their first job.
    'No unemployment': dict(job_fin_prob=1, job_sep_prob=0),
}


results = {}
for name, overrides in scenarios.items():
    out_s = dataproject.simulate(seed=2025, N=50_000, **overrides)

    # Gini for the full sample
    gini_pooled = dataproject.gini(out_s['income'].flatten())

    # Gini separately for each age
    gini_age = dataproject.gini_by_age(out_s['income'], out_s['ages'])

    results[name] = {'pooled': gini_pooled, 'by_age': gini_age}

# Summary table: pooled Gini for each scenario, compared to the baseline
baseline_pooled = results['Baseline']['pooled']

print(f'{'Scenario':<32}{'Gini (pooled)':>15}{'Diff. vs baseline':>20}')
for name, res in results.items():
    diff = res['pooled'] - baseline_pooled
    print(f'{name:<32}{res['pooled']:>15.3f}{diff:>20.3f}')

# Plot: Gini by age, one line per scenario
plt.figure(figsize=(8, 5))
for name, res in results.items():
    df_age = res['by_age']
    plt.plot(df_age['age'], df_age['gini'], label=name)

plt.xlabel('Age')
plt.xticks(df_age['age'][::5], rotation=45)
plt.ylabel('Gini coefficient')
plt.title('Gini coefficient by age, baseline vs. mechanisms switched off')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

We find that **human capital shocks matter the most**. Removing the human capital shock gives the largest drop in pooled Gini and by far the lowest by-age curve. **Educational differences** matter almost as much by widening both starting levels and growth rates of human capital. **Depreciation while unemployed** and **unemployment itself** barely affect within-age inequality, however both slightly raise the pooled Gini since removing them lets human capital grow uninterrupted, stretching the life-cycle income gap between young and old even as same-age inequality stays flat. This is exactly the divergence the hint points to: the pooled measure mixes within-age and across-age (life-cycle) dispersion, so a mechanism can leave one unchanged while still shifting the other.

### Question 2.5: Extension: More Risk <a id='toc2_5_'></a>

We extent the model by adding more risk through a higher job-seperation probability for people with a lower education, compared to people with a higher education. This can be viewed as adding business cycles, where there is higher risk of unemployment for people with lower education levels compared to higher, that usually have more stable jobs. To do this, we have added a new job seperation rate, where $job\_sep\_prob\_i=0.05+U(0, 0.30) \cdot \frac{1}{S_i}$, so that there in some periods, where there is a much higher risk for people with a lower education of losing their job. This is illustrated by the uniform distribution, where each individual in each period pulls an 'added risk' of losing their employment.

In [ ]:
# Run the simulation with the added risk
out_more_risk = dataproject.simulate(seed=2025, N=50_000, sep_shock_max=0.30, more_risk=True)

# Compare the gini coefficient with more risk to baseline:
gini_pooled_baseline = dataproject.gini(out['income'].flatten())
gini_pooled_more_risk = dataproject.gini(out_more_risk['income'].flatten())

print(f'Gini (pooled), baseline:  {gini_pooled_baseline:.3f}')
print(f'Gini (pooled), more risk: {gini_pooled_more_risk:.3f}')
print(f'Adding more risk to the model yields a increase of Gini by: {gini_pooled_more_risk - gini_pooled_baseline:.3f}')

gini_age_baseline = dataproject.gini_by_age(out['income'], out['ages'])
gini_age_more_risk = dataproject.gini_by_age(out_more_risk['income'], out_more_risk['ages'])

We now look at the Gini Coefficient for the single age groups.

In [ ]:
# Plot the Gini coefficient by age for both scenarios:
plt.figure(figsize=(8, 5))

plt.plot(gini_age_baseline['age'], gini_age_baseline['gini'], label='Baseline')
plt.plot(gini_age_more_risk['age'], gini_age_more_risk['gini'], label='More risk (education-scaled)')

plt.xlabel('Age')
plt.xticks(gini_age_baseline['age'][::5], rotation=45)
plt.ylabel('Gini coefficient')
plt.title('Gini coefficient by age: baseline vs. education-scaled separation risk')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

As in the pooled case, the Gini Coefficient increases, especially in the older age groups, compared to the base line model. 

In [ ]:
#Look at how it affects the income distribution, as we did in 2.2.3
target_ages = [25, 35, 45, 60]
target_indices = [a - 18 for a in target_ages] 

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
xmax = max(out['income'][target_indices, :].max(), out_more_risk['income'][target_indices, :].max())

for ax, age, t in zip(axes, target_ages, target_indices):
    ax.hist(out_more_risk['income'][t, :], bins=100, range=(0, xmax), color='indianred',
             alpha=0.6, label='More risk', zorder=1)
    ax.hist(out['income'][t, :], bins=100, range=(0, xmax), color='steelblue',
             alpha=0.6, label='Baseline', zorder=2)
    ax.set_title(f'Age {age}')
    ax.set_xscale('log')
    ax.set_xlabel('Income')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

We find that by adding more risk, the gini coefficient becomes slightly higher compared to baseline. This is because the education-scaled job-separation risk falls disproportionately on individuals with less education, who therefore experience unemployment more often. Each unemployment spell leads to human capital depreciation and lower income while it lasts. This effect also trickles in to the income distribution, where we see that by adding more risk, more people are placed in the lower bins of the income distribution. That is because the extra risk falls disproportionatly on the lower-educated, pushing them into unemployment and therefore a lower income. The gap between the two scenarios grows with age, since the extra risk is drawn every period and therefore compounds over the life cycle.